In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
import math
from tqdm import tqdm

In [ ]:
#pd.set_option("display.max_rows", None)

# Prepare metadata

In [ ]:
meta = pd.read_csv("/mnt/custom-file-systems/efs/fs-0d3b03a3638d516f0_fsap-05c6d9e300aefa0e9/projects/mofa/mofa_workflow/input_data_mosaic/Prepared_Sample_Meta_Data.csv")

In [ ]:
meta

In [ ]:
meta['sample_id'] = meta['sample_id'].str.replace('_', '')

In [ ]:
meta

In [ ]:
del meta["Unnamed: 0"]

In [ ]:
meta.to_csv("/mnt/custom-file-systems/efs/fs-0d3b03a3638d516f0_fsap-05c6d9e300aefa0e9/projects/mofa/mofa_workflow/input_data_mosaic/Prepared_Sample_Meta_Data.csv", index=False)

In [ ]:
meta["number_of_brain_tumour_sites"] = meta["number_of_brain_tumour_sites"].astype(float)

In [ ]:
meta.to_csv("/mnt/custom-file-systems/efs/fs-0d3b03a3638d516f0_fsap-05c6d9e300aefa0e9/projects/mofa/mofa_workflow/input_data_mosaic/Prepared_Sample_Meta_Data.csv", index=False)

In [ ]:
meta = pd.read_csv("/mnt/custom-file-systems/efs/fs-0d3b03a3638d516f0_fsap-05c6d9e300aefa0e9/projects/mofa/mofa_workflow/input_data_mosaic/Prepared_Sample_Meta_Data.csv")

In [ ]:
# Replace with pandas' native NA value
meta.replace("Unknown", np.nan, inplace=True)

In [ ]:
meta

In [ ]:
meta["number_of_brain_tumour_sites"] = meta["number_of_brain_tumour_sites"].astype(int)

In [ ]:
meta["number_of_brain_tumour_sites"] = meta["number_of_brain_tumour_sites"].astype(str)

In [ ]:
mapping = {"1": "one", "2": "two", "3": "three"}
meta["number_of_brain_tumour_sites"] = meta["number_of_brain_tumour_sites"].replace(mapping)

In [ ]:
meta["number_of_brain_tumour_sites"]

In [ ]:
meta.to_csv("resources/Prepared_Sample_Meta_Data_cleaned.csv", index=False)

In [ ]:
meta["smoking_status"]

# prepare pathways

In [ ]:
df = pd.read_csv("resources/h.all.v2024.1.Hs.symbols.gmt", sep="\t", header=None)

In [ ]:
# Identify the gene columns 
gene_cols = df.columns[2:]

# Melt the dataframe so that each gene becomes a row
df_long = df.melt(id_vars=df.columns[:2], 
                  value_vars=gene_cols, 
                  var_name='gene_num', 
                  value_name='gene')

# Remove rows where 'gene' is NaN
df_long = df_long.dropna(subset=['gene'])

# Rename the identifier columns for clarity 
df_long = df_long.rename(columns={df.columns[0]: 'pathway_name', 
                                  df.columns[1]: 'ID'})

# Reset the index and create a new sequential index column (starting at 1)
df_long = df_long.reset_index(drop=True)
df_long.index = df_long.index + 1  # to start index at 1
df_long = df_long.reset_index().rename(columns={'index': 'index'})

In [ ]:
df_long = df_long.drop(columns=["gene_num", "index"])
df_long['pathway_name'] = df_long['pathway_name'].str.replace('_', '-', regex=False)
df_long["ID"] = df_long["pathway_name"]

In [ ]:
df_long = df_long[['ID', 'gene', 'pathway_name']]

In [ ]:
df_long

In [ ]:
df_long.to_csv("resources/pathways_hallmark_cancer.csv")